In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
df = pd.read_csv(path + '/Q1_data.csv')


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
df['Delivery_Time'].plot()

In [ ]:
#another plot
df['Delivery_Time'].hist()

In [ ]:
# Task 1: Write your code here:
print(df.columns)
df = df.drop('Order_ID', axis= 1)
print(df.columns)


In [ ]:
print(df['Weather'].mode().values[0])

In [ ]:
# Task 2: Write your code here:
print(df.isna().sum())
for col in df.select_dtypes(exclude= 'object').columns:
  df[col] = df[col].fillna(df[col].mean().astype(df[col].dtype))

for col in df.select_dtypes(include= 'object').columns:
  df[col] = df[col].fillna(df['Weather'].mode().values[0])
print('after -- \n')
print(df.isna().sum())

In [ ]:
df.isna().sum().sum()

In [ ]:
# Task 3: Write your code here:
print(df.duplicated().sum())

df = df.drop_duplicates()

print(df.duplicated().sum())

In [ ]:
df.head()

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder

#i've written it while studiyng
def use_onehotencoder(df: pd.DataFrame) -> pd.DataFrame:
    #create object
    # drop first is used to make linear regression work without collinearity (it will remove the first class), sparse output false to convert into df later
    #drop first makes it when indicating the first class place all classes 0 as a value in it
    oh = OneHotEncoder(drop= 'first', sparse_output=False)

    #take only obj columns to place in the object
    cats = df.select_dtypes(include= 'object').columns

    #convert them into encoded values
    encoded_values = oh.fit_transform(df[cats])

    #convert the encoded values into dataframes, the names could be automatically placed using get_feature_names_out of the object place the same index of df as we will concat them
    encoded_df = pd.DataFrame(encoded_values, columns= oh.get_feature_names_out(), index= df.index)

    #concat them and remove old obj columns ***do not forget [] ***to place dfs
    new_encoded_df = pd.concat([df.drop(cats, axis= 1), encoded_df], axis= 1)

    return new_encoded_df

df_encoded = use_onehotencoder(df).head()


In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler
#split to ttarget and features
X = df_encoded.drop('Delivery_Time', axis= 1)
y = df_encoded['Delivery_Time']

s_scaler = StandardScaler()
X_scaled = s_scaler.fit_transform(X)
print(type(X_scaled))
#convert to pandas
X_scaled = pd.DataFrame(X_scaled, columns=s_scaler.get_feature_names_out(), index= X.index)
X_scaled.head()



In [ ]:
# Task 6: Write your code here:
print(df['Delivery_Time'].dtype)

# we dont need to check as it is regression not classification

In [ ]:
# Task 1: Write your code here:

# i've splittet it before so i will just reassign them to same variable names
X = X_scaled
y = y

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
# Task 2,3,4,5: Write your code here:
# we don't need stratify as it is regression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
import numpy as np


kf = KFold(n_splits= 5, random_state= 42, shuffle= True)

all_error = 0
number_of_models = 0

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
for i, (train_index, test_index) in enumerate(kf.split(X, y)):
    X_Train, X_Test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    #random forest regressor
    model.fit(X_Train, y_train)
    preds = model.predict(X_Test)

    MAE = mean_absolute_error(y_test, preds)
    all_error +=MAE
    number_of_models += 1

print(f"all error sum = {all_error}")
print(f"num of models = {number_of_models}")

avg = all_error/number_of_models

print(f'avrage is -> {avg}')



In [ ]:
# Task 1: Write your code here:
#last model iteration's will be used to get feature importance
import matplotlib.pyplot as plt
importance = pd.Series(model.feature_importances_, index = X.columns )
#sort it
importance.sort_values(ascending= False, inplace= True)
#plot feature importances
plt.figure(figsize=(10, 6))
plt.barh(importance.index, importance.values,  color='red')
plt.xlabel('Coefficient Value')
plt.ylabel('Features')
plt.title('last model importance')
plt.show()



In [ ]:
preds

In [ ]:
# Task 2: Write your code here:
# plot the preds of the last iteration
plt.hist(preds)


In [ ]:
# Task Bonus: Write your code here:
from catboost import CatBoostRegressor

kf = KFold(n_splits= 5, random_state= 42, shuffle= True)

all_error = 0
number_of_models = 0

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model_2 = CatBoostRegressor(verbose=0)
for i, (train_index, test_index) in enumerate(kf.split(X, y)):
    X_Train, X_Test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    #random forest regressor
    model.fit(X_Train, y_train)
    model_2.fit(X_Train, y_train)
    preds = model.predict(X_Test)
    preds_2 = model_2.predict(X_Test)

    avg = (preds+preds_2)/2

    MAE = mean_absolute_error(y_test, avg)
    all_error +=MAE
    number_of_models += 1

print(f"all error sum = {all_error}")
print(f"num of models = {number_of_models}")

avg = all_error/number_of_models

print(f'avrage is -> {avg}')

